# Fiber flux delivery ratio vs. linphi positioners

Third companion in this family (`linphi_splitflux.ipynb`, `calibstars_linphi.ipynb`), from
a diagnostic the data-systems team has used to check focus (large radial trends in this
ratio across the focal plane indicate being out of focus):

```python
im = '00126561'; night = '20220317'
from astropy.table import Table
for spec in range(10):
    t = Table.read(f".../cframe-r{spec}-{im}.fits.gz", "FIBERMAP")
    s = Table.read(f".../cframe-r{spec}-{im}.fits.gz", "SCORES")
    i = t["FIBERFLUX_R"] > 10          # imaging flux, only sufficiently bright sources
    ratio = s["MEDIAN_CALIB_COUNT_R"][i] / t["FIBERFLUX_R"][i]
    ratio /= np.median(ratio)
    plt.scatter(t["FIBER_X"][i], t["FIBER_Y"][i], c=ratio, vmin=0.8, vmax=1.2)
```

`FIBERFLUX_R` (from `FIBERMAP`, i.e. imaging/Legacy-Survey photometry -- the "true" flux a
perfectly-centered fiber should have captured) vs. `MEDIAN_CALIB_COUNT_R` (from `SCORES`,
what the fiber actually delivered): the ratio is "did we get the flux we expected." Unlike
`calibstars_linphi.ipynb`, this is **not restricted to standard stars** -- any object bright
enough (`FIBERFLUX_R > 10`) qualifies, which is a much larger per-exposure sample (typically
hundreds to low thousands of fibers per exposure vs. ~100-300 standard stars), directly
helping the linphi-side statistics.

Two deliberate differences from the data-systems snippet, for this comparison specifically:
- **No `ratio /= median(ratio)` step.** The original code's per-exposure median-normalization
  is designed to isolate *spatial* (radial/focus) residuals within one exposure by removing
  the overall calibration scale. For a regular-vs-linphi comparison, that per-exposure scale
  factor is common to both groups, so it shouldn't change *whether* they differ -- but
  dropping it keeps the raw, physically-meaningful ratio, at the cost of pooling in real
  exposure-to-exposure variation (transparency, seeing) that median-normalizing would have
  absorbed. Worth watching for in the results: if between-exposure scatter dominates, that's
  this design choice showing up, not a linphi effect.
- **PETAL_LOC/DEVICE_LOC come directly from the same FIBERMAP** (`t["PETAL_LOC"]`,
  `t["DEVICE_LOC"]`) -- no join needed, unlike `calibstars`'s FIBER-only index.

**Performance note (updated 2026-07-17)**: this needs `MEDIAN_CALIB_COUNT_R` from `SCORES`,
which only lives in `cframe_table` -- `fiberassign_table`'s ~60x speedup doesn't apply here,
since fiberassign has no measured-flux columns at all. Still 10 cframe reads per exposure
(r-arm only, since everything here is R-band). An initial run of this notebook measured
~20s/exposure, which led to two real fixes:

1. `cframe_table` was opening each cframe file *twice* (once for `FIBERMAP`, once for
   `SCORES`), each open re-paying the file's gzip decompression cost -- fixed to read both
   from one open, ~30% faster per file (a fair A/B measurement; an earlier same-file timing
   comparison had suggested ~8x, but that number was inflated by OS page-cache warm-up bias).
2. Even with that fix, each file read is still dominated by gzip decompression, since
   `FIBERMAP`/`SCORES` are the *last* two extensions in the file, after the much larger
   pixel-array extensions, and gzip isn't seekable -- reaching them costs close to a
   full-file decompression regardless. That decompression is CPU-bound C code that doesn't
   release the GIL, so threads don't help, but real OS processes do: `Exposure.cframe_tables`
   (new) reads several cameras in parallel processes and measured ~3.8x faster than looping
   `cframe_table` sequentially (~0.4s/camera vs. ~1.4s/camera, 10 r-cameras). This notebook's
   per-exposure loop now uses it instead of looping `cframe_table` one petal at a time.

Combined, that's roughly a 5x improvement over the original ~20s/exposure baseline -- still
the slowest of the three sibling notebooks (no way around needing 10 per-petal cframe
files), but no longer as painful to scale up. Start small (see the exposure-selection cell)
before scaling up.

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Add telemetry_mining to the path. Point DOS_TELEMETRY_MINING_DIR at your own
# checkout (svn co https://desi.lbl.gov/svn/code/online/telemetry_mining/trunk);
# defaults to the maintainer's checkout at NERSC.
TM_DIR = os.getenv("DOS_TELEMETRY_MINING_DIR", os.path.expanduser("~/telemetry_mining-trunk/src"))
sys.path.insert(0, TM_DIR)
from telemetry_mining import Exposure, select_exposures, harvest

## Exposure selection

Same primitive as `calibstars_linphi.ipynb` (`select_exposures`, no same-tile constraint
here either -- every fiber measurement stands on its own). Starting with a small `EXPIDS`
list to validate correctness first, given the slower per-exposure cost noted above --
widen to a `NIGHT_RANGE` once this looks right, the same way `calibstars_linphi.ipynb` was
scaled up.

In [ ]:
# Set exactly one of these -- leave the other as None
NIGHT_RANGE = [20260501, 20260520]                           # e.g. (20260101, 20260710), inclusive
EXPIDS = None             # or an explicit list, e.g. a split triplet [255008, 255009, 255010]

SEQUENCE = ['DESI']   # 'DESI' alone drops '_Split' follow-ups -- add '_Split' to include them
MIN_TEFF = 30.0       # minimum accumulated effective time (s), from the ETC's real-time totteff
MIN_FIBERFLUX_R = 10.0  # imaging flux threshold -- matches the data-systems snippet

assert (NIGHT_RANGE is None) != (EXPIDS is None), "set exactly one of NIGHT_RANGE or EXPIDS, not both/neither"

if NIGHT_RANGE is not None:
    where = "sequence = ANY(%s) and night between %s and %s and totteff > %s"
    params = (SEQUENCE, NIGHT_RANGE[0], NIGHT_RANGE[1], MIN_TEFF)
else:
    where = "sequence = ANY(%s) and id = ANY(%s) and totteff > %s"
    params = (SEQUENCE, list(EXPIDS), MIN_TEFF)

expids = list(select_exposures(where, params=params)['EXPID'])
print(f'{len(expids)} exposures selected')


## Per exposure: ratio + linphi flag, all 10 petals (r-arm only)

For each exposure, fetch all 10 `r{petal}` cameras at once via `cframe_tables` (not all 30
-- everything here is R-band), which reads them in parallel processes rather than looping
`cframe_table` one petal at a time -- see the intro cell's Performance note. Filter to
`FIBERFLUX_R > 10`, compute the ratio, and attach `POS_LINPHI` from `exp.coords` (same join
key/convention as the sibling notebooks). `PETAL_LOC`/`DEVICE_LOC` come straight from the
combined table's own index -- no join needed.

In [ ]:
def bright_fiber_ratio(e):
    """Per-exposure: MEDIAN_CALIB_COUNT_R / FIBERFLUX_R for bright fibers, with the POS_LINPHI flag.
    Returns None to skip (no coords file, or no bright fibers) -- harvest drops those."""
    try:
        linphi = e.coords[['POS_LINPHI']]
    except Exception:
        return None
    # cframe_tables reads all 10 r-camera petals in one process-parallel batch; `errors` holds any
    # pruned/missing petal files (just fewer petals contribute, not a reason to skip the exposure).
    tables, errors = e.cframe_tables([f'r{petal}' for petal in range(10)])
    frames = []
    for combined in tables.values():
        bright = combined[combined['FIBERFLUX_R'] > MIN_FIBERFLUX_R].copy()
        if bright.empty:
            continue
        bright['ratio'] = bright['MEDIAN_CALIB_COUNT_R'] / bright['FIBERFLUX_R']
        frames.append(bright[['ratio']])
    if not frames:
        return None
    return pd.concat(frames).join(linphi)   # PETAL_LOC/DEVICE_LOC is the shared index


# harvest pools ONE DataFrame across all exposures (bulk `night`, `EXPID` column, None-skip).
# NOTE: no max_workers here -- cframe_tables already parallelizes *within* an exposure with a
# process pool, and nesting that under a thread pool would oversubscribe the filesystem (the
# cframe_tables "don't nest with cross-exposure parallelism" caveat). calibstars_linphi, whose fn
# has no internal pool, does use max_workers.
ratio_all = harvest(expids, bright_fiber_ratio, concat=True)
print(f'{len(ratio_all)} bright-fiber measurements across {ratio_all["EXPID"].nunique()} exposures')


## Regular vs. linphi robots: ratio comparison

Same string-vs-bool gotcha as the sibling notebooks: `POS_LINPHI` is the literal string
`'True'`/`'False'`, not a real boolean. No median-normalization here (see intro) -- watch
the raw scale, and treat any exposure-to-exposure spread as expected, not a bug.

In [ ]:
regular = ratio_all[ratio_all['POS_LINPHI'] == 'False']
regular = regular[regular['ratio']<5.0]
linphi = ratio_all[ratio_all['POS_LINPHI'] == 'True']
linphi = linphi[linphi['ratio']<5.0]
print(f'{len(regular)} regular-robot measurements, {len(linphi)} linphi-robot measurements')
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.hist(regular['ratio'], bins=100, range=(0, 2), color='b', alpha=0.6)
plt.title(f'MEDIAN_CALIB_COUNT_R / FIBERFLUX_R (regular robots), n={len(regular)}',fontsize=8)
plt.subplot(1, 2, 2)
plt.hist(linphi['ratio'], bins=100, range=(0, 2), color='r', alpha=0.6)
plt.title(f'MEDIAN_CALIB_COUNT_R / FIBERFLUX_R (linphi robots), n={len(linphi)}',fontsize=8)
plt.show()

print(f"mean ratio: regular={regular['ratio'].mean():.4f}, linphi={linphi['ratio'].mean():.4f}")
print(f"std  ratio: regular={regular['ratio'].std():.4f}, linphi={linphi['ratio'].std():.4f}")

## Recap

| Question | How |
|---|---|
| Bigger sample than `calibstars_linphi.ipynb`? | Yes -- any `FIBERFLUX_R > 10` fiber, not just ~100-300 standard stars per exposure |
| PETAL_LOC/DEVICE_LOC for this table? | Already the `cframe_table` index -- no join needed, unlike `calibstars`'s FIBER-only index |
| Median-normalized like the data-systems snippet? | No, deliberately -- see the intro cell for the tradeoff |

### Result (2026-07-17, `NIGHT_RANGE = [20260501, 20260520]`, 294 exposures, 268 processed)

Two real, worth-noting complications found running this at scale, both fixed:

- **Unbounded-ratio outliers.** Unlike `calibstars_linphi.ipynb`'s `RCALIBFRAC` (already a
  bounded, calibrated ratio), `MEDIAN_CALIB_COUNT_R / FIBERFLUX_R` is unbounded and, without
  a point-source cut (this notebook deliberately includes extended sources, unlike
  `linphi_splitflux.ipynb`'s `MORPHTYPE == 'PSF'` cut), ran as high as 128 pre-cut. Fixed
  with an empirical `ratio < 5.0` cut, applied to both groups.
- **A filtering-order bug**, caught by inspection before trusting the first pass: an earlier
  version filtered `linphi` and then immediately overwrote it unfiltered on the next line
  (worked in a reused Jupyter kernel where a stale `linphi` masked the bug; would have
  failed loudly on a fresh restart). Fixed by assigning both groups from `ratio_all` first,
  then filtering both symmetrically.

|                | regular robots | linphi robots |
|---|---|---|
| n measurements (post-cut) | 177,480 | 8,547 |
| mean ratio | 0.4320 | 0.4345 |
| std ratio  | 0.1103 | 0.1393 |

**Mean essentially identical; linphi robots show ~26% more scatter.** This closely matches
`calibstars_linphi.ipynb`'s independent result (~27% more scatter, no mean offset, from a
different quantity -- `RCALIBFRAC` on standard stars only vs. this notebook's raw ratio over
any sufficiently bright object) -- two different methods converging on the same conclusion.
A small (14-exposure), quick isolated timing/correctness check on a different night range
(`[20260501, 20260503]`) showed the *opposite* std ordering by chance, exactly the
exposure-to-exposure confound flagged in the intro cell for small, unnormalized samples --
underscoring that this design only becomes reliable at the larger scale used here.

**Bottom line across both notebooks**: linphi-affected positioners are measurably less
precise than regular ones (more scatter, not a directional bias) but still deliver
reasonable science-quality data on average -- not a severe fault, a real but modest
degradation.

### Reproducibility check (2026-07-17, same `NIGHT_RANGE`, re-run with `cframe_tables`)

Re-ran the same 294-exposure selection after switching the per-exposure loop from looping
`cframe_table` one petal at a time to the new parallel `Exposure.cframe_tables` (see the
intro cell's Performance note) -- felt noticeably faster, and produced **exactly the same
result**: 186,069 total measurements across the same 268 processed / 26 skipped exposures,
identical 177,480/8,547 regular/linphi split, identical mean/std to four decimal places.
Good independent confirmation that the parallel reader is a pure performance change, not a
correctness change.